# Deep Agents - 진행 상태 추적 미들웨어

## 개요

Deep Agents는 복잡한 작업을 처리하기 위해 설계된 LangChain의 고급 에이전트 시스템입니다.
이 노트북에서는 Deep Agents의 미들웨어 아키텍처를 활용하여 진행 상태를 추적하는 시스템을 구축합니다.

## Deep Agents란?

Deep Agents는 다음 기능을 제공하는 강력한 에이전트 시스템입니다:

1. **TodoListMiddleware**: 작업 계획 및 추적
2. **FilesystemMiddleware**: 단기/장기 메모리 관리
3. **SubAgentMiddleware**: 서브 에이전트 생성 및 위임

## 학습 목표

- `create_deep_agent`를 사용한 Deep Agent 생성
- 커스텀 미들웨어 구현 및 통합
- 진행 상태 실시간 추적
- 여러 미들웨어 조합

## 환경 설정

In [ ]:
# 필요한 라이브러리 설치
# !pip install langchain langchain-anthropic langgraph deepagents

In [ ]:
import os
from datetime import datetime
from typing import Dict, List, Any, Optional
from collections import defaultdict
import asyncio
import time

# Deep Agents 핵심 import
from deepagents import create_deep_agent
from deepagents.middleware import TodoListMiddleware, FilesystemMiddleware, SubAgentMiddleware
from deepagents.middleware.base import Middleware

from langchain.callbacks.base import AsyncCallbackHandler, BaseCallbackHandler
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_anthropic import ChatAnthropic

# API 키 설정
# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"

print("✅ Deep Agents 라이브러리 import 완료")

## 1. Deep Agent 기본 이해

먼저 기본 Deep Agent를 생성하고 어떻게 작동하는지 이해합니다.

In [ ]:
# 기본 도구 정의
@tool
def search_web(query: str) -> str:
    """웹을 검색합니다.
    
    Args:
        query: 검색 쿼리
    """
    time.sleep(0.5)
    return f"'{query}'에 대한 검색 결과: 관련 정보 5개 발견"


@tool
def write_document(content: str, filename: str) -> str:
    """문서를 작성합니다.
    
    Args:
        content: 문서 내용
        filename: 파일명
    """
    time.sleep(0.3)
    return f"문서 '{filename}' 작성 완료 ({len(content)}자)"


print("✅ 기본 도구 정의 완료")

In [ ]:
# 기본 Deep Agent 생성
basic_agent = create_deep_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_web, write_document]
)

print("✅ 기본 Deep Agent 생성 완료")
print("\n기본적으로 포함된 미들웨어:")
print("  - TodoListMiddleware: 작업 계획 및 추적")
print("  - FilesystemMiddleware: 파일 시스템 관리")
print("  - SubAgentMiddleware: 서브 에이전트 관리")

## 2. 진행 상태 추적 미들웨어 구현

Deep Agents의 미들웨어 시스템을 확장하여 진행 상태를 추적하는 커스텀 미들웨어를 만듭니다.

In [ ]:
class ProgressTrackingMiddleware(Middleware):
    """Deep Agent용 진행 상태 추적 미들웨어
    
    Deep Agents의 Middleware 베이스 클래스를 상속받아
    에이전트의 작업 진행 상태를 실시간으로 추적합니다.
    """
    
    def __init__(self, handlers: Optional[List[BaseCallbackHandler]] = None):
        super().__init__()
        self.handlers = handlers or []
        self.progress_data = {
            'tasks': [],
            'completed': 0,
            'total': 0,
            'current_task': None,
            'start_time': datetime.now()
        }
    
    def add_handler(self, handler: BaseCallbackHandler):
        """핸들러 추가"""
        self.handlers.append(handler)
    
    async def on_task_start(self, task_name: str, task_id: str):
        """작업 시작 시 호출"""
        self.progress_data['current_task'] = task_name
        self.progress_data['tasks'].append({
            'id': task_id,
            'name': task_name,
            'status': 'in_progress',
            'start_time': datetime.now().isoformat()
        })
        
        await self._notify_handlers('task_start', {
            'task_name': task_name,
            'task_id': task_id
        })
    
    async def on_task_complete(self, task_id: str):
        """작업 완료 시 호출"""
        for task in self.progress_data['tasks']:
            if task['id'] == task_id:
                task['status'] = 'completed'
                task['end_time'] = datetime.now().isoformat()
                break
        
        self.progress_data['completed'] += 1
        
        # 진행률 계산
        if self.progress_data['total'] > 0:
            percent = (self.progress_data['completed'] / self.progress_data['total']) * 100
        else:
            percent = 0
        
        await self._notify_handlers('task_complete', {
            'task_id': task_id,
            'completed': self.progress_data['completed'],
            'total': self.progress_data['total'],
            'percent': percent
        })
    
    async def on_todos_updated(self, todos: List[Dict]):
        """TodoListMiddleware의 todos가 업데이트될 때 호출"""
        self.progress_data['total'] = len(todos)
        completed = sum(1 for t in todos if t.get('status') == 'completed')
        self.progress_data['completed'] = completed
        
        percent = (completed / len(todos) * 100) if len(todos) > 0 else 0
        
        await self._notify_handlers('progress_update', {
            'todos': todos,
            'completed': completed,
            'total': len(todos),
            'percent': int(percent),
            'timestamp': datetime.now().isoformat()
        })
    
    async def _notify_handlers(self, event_type: str, data: Dict[str, Any]):
        """등록된 핸들러에 이벤트 전달"""
        event = {'type': event_type, **data}
        
        for handler in self.handlers:
            if hasattr(handler, 'on_progress_event'):
                if asyncio.iscoroutinefunction(handler.on_progress_event):
                    await handler.on_progress_event(event)
                else:
                    handler.on_progress_event(event)
    
    def get_progress_summary(self) -> Dict[str, Any]:
        """진행 상태 요약 반환"""
        elapsed = (datetime.now() - self.progress_data['start_time']).total_seconds()
        
        return {
            'elapsed_time': elapsed,
            'completed': self.progress_data['completed'],
            'total': self.progress_data['total'],
            'percent': (self.progress_data['completed'] / self.progress_data['total'] * 100) 
                      if self.progress_data['total'] > 0 else 0,
            'current_task': self.progress_data['current_task'],
            'tasks': self.progress_data['tasks']
        }


print("✅ ProgressTrackingMiddleware 정의 완료")

## 3. 진행 상태 대시보드 핸들러

미들웨어에서 발생한 이벤트를 받아 대시보드에 표시하는 핸들러입니다.

In [ ]:
class ProgressDashboardHandler(BaseCallbackHandler):
    """진행 상태 대시보드 핸들러"""
    
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.event_history = []
    
    def on_progress_event(self, event: Dict[str, Any]):
        """진행 상태 이벤트 처리"""
        self.event_history.append(event)
        
        if not self.verbose:
            return
        
        event_type = event.get('type')
        
        if event_type == 'task_start':
            self._display_task_start(event)
        elif event_type == 'task_complete':
            self._display_task_complete(event)
        elif event_type == 'progress_update':
            self._display_progress_update(event)
    
    def _display_task_start(self, event: Dict[str, Any]):
        """작업 시작 표시"""
        print(f"\n🚀 작업 시작: {event['task_name']}")
        print(f"   작업 ID: {event['task_id']}")
    
    def _display_task_complete(self, event: Dict[str, Any]):
        """작업 완료 표시"""
        print(f"\n✅ 작업 완료: {event['task_id']}")
        print(f"   진행률: {event['percent']:.1f}% ({event['completed']}/{event['total']})")
    
    def _display_progress_update(self, event: Dict[str, Any]):
        """진행 상태 업데이트 표시"""
        print(f"\n{'='*70}")
        print(f"📊 [Progress Dashboard] 진행 상태 업데이트")
        print(f"{'='*70}")
        print(f"⏱️  시간: {event.get('timestamp', 'N/A')}")
        print(f"📈 진행률: {event['percent']}% ({event['completed']}/{event['total']})")
        
        # 진행률 바
        bar_length = 50
        filled = int(bar_length * event['completed'] / event['total']) if event['total'] > 0 else 0
        bar = "█" * filled + "░" * (bar_length - filled)
        print(f"\n[{bar}] {event['percent']}%\n")
        
        # 할 일 목록
        if event.get('todos'):
            print("📝 작업 목록:")
            for i, todo in enumerate(event['todos'], 1):
                status = todo.get('status', 'pending')
                icon = {
                    'completed': '✅',
                    'in_progress': '🔄',
                    'pending': '⏳',
                    'cancelled': '❌'
                }.get(status, '⚪')
                content = todo.get('content', 'N/A')
                print(f"  {i}. {icon} {content} [{status}]")
        
        print(f"{'='*70}\n")
    
    def get_summary(self) -> str:
        """진행 상태 요약"""
        if not self.event_history:
            return "이벤트 없음"
        
        progress_events = [e for e in self.event_history if e.get('type') == 'progress_update']
        if not progress_events:
            return "진행 상태 업데이트 없음"
        
        latest = progress_events[-1]
        return f"""\n진행 상태 요약:
  - 총 이벤트: {len(self.event_history)}개
  - 현재 진행률: {latest['percent']}%
  - 완료된 작업: {latest['completed']}/{latest['total']}
"""


print("✅ ProgressDashboardHandler 정의 완료")

## 4. Deep Agent with Progress Tracking

이제 진행 상태 추적 미들웨어를 포함한 Deep Agent를 생성합니다.

In [ ]:
# 작업 도구들
@tool
def research_topic(topic: str) -> str:
    """주제를 조사합니다.
    
    Args:
        topic: 조사할 주제
    """
    time.sleep(1)
    return f"'{topic}' 조사 완료: 10개의 참고 자료 수집, 주요 발견사항 3가지 정리됨"


@tool
def analyze_data(data_source: str) -> str:
    """데이터를 분석합니다.
    
    Args:
        data_source: 분석할 데이터 소스
    """
    time.sleep(1.2)
    return f"'{data_source}' 데이터 분석 완료: 긍정적 트렌드 발견, 3개의 인사이트 도출"


@tool
def create_report(title: str, sections: int) -> str:
    """리포트를 생성합니다.
    
    Args:
        title: 리포트 제목
        sections: 섹션 수
    """
    time.sleep(0.8)
    return f"리포트 '{title}' 생성 완료: {sections}개 섹션, 15페이지"


@tool
def review_content(content_type: str) -> str:
    """컨텐츠를 검토합니다.
    
    Args:
        content_type: 검토할 컨텐츠 유형
    """
    time.sleep(0.6)
    return f"'{content_type}' 검토 완료: 개선 제안 5개, 승인 권장"


print("✅ 작업 도구 정의 완료")

In [ ]:
# 진행 상태 추적이 포함된 Deep Agent 생성
dashboard = ProgressDashboardHandler(verbose=True)
progress_middleware = ProgressTrackingMiddleware(handlers=[dashboard])

# Deep Agent 생성 with 커스텀 미들웨어
agent = create_deep_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[research_topic, analyze_data, create_report, review_content],
    # 기본 미들웨어 + 커스텀 미들웨어
    middleware=[
        TodoListMiddleware(
            system_prompt="""복잡한 작업을 수행할 때:
1. write_todos 도구로 단계별 계획을 수립하세요
2. 각 작업을 순차적으로 실행하세요
3. 작업 완료 시 write_todos를 호출하여 상태를 'completed'로 업데이트하세요
"""
        ),
        progress_middleware,  # 우리가 만든 진행 상태 추적 미들웨어
    ]
)

print("✅ 진행 상태 추적 Deep Agent 생성 완료")
print("\n포함된 미들웨어:")
print("  1. TodoListMiddleware (기본)")
print("  2. ProgressTrackingMiddleware (커스텀)")

## 5. Deep Agent 실행 및 진행 상태 추적

실제 작업을 수행하면서 진행 상태를 실시간으로 추적합니다.

In [ ]:
# TodoList와 통합한 시뮬레이션
def simulate_deep_agent_workflow():
    """Deep Agent 워크플로우 시뮬레이션"""
    
    print("🚀 Deep Agent 작업 시작\n")
    print("작업: AI 기술 트렌드 분석 리포트 작성\n")
    
    # 작업 계획 (TodoList 시뮬레이션)
    todos = [
        {"id": "1", "content": "AI 기술 트렌드 조사", "status": "pending"},
        {"id": "2", "content": "시장 데이터 분석", "status": "pending"},
        {"id": "3", "content": "리포트 작성", "status": "pending"},
        {"id": "4", "content": "최종 검토", "status": "pending"},
    ]
    
    # 초기 진행 상태
    asyncio.run(progress_middleware.on_todos_updated(todos))
    time.sleep(1)
    
    # 작업 1: AI 기술 트렌드 조사
    print("\n" + "="*70)
    print("📋 작업 1/4: AI 기술 트렌드 조사")
    print("="*70)
    
    asyncio.run(progress_middleware.on_task_start("AI 기술 트렌드 조사", "task_1"))
    result = research_topic.invoke({"topic": "2024 AI 기술 트렌드"})
    print(f"\n{result}")
    
    todos[0]['status'] = 'completed'
    asyncio.run(progress_middleware.on_task_complete("task_1"))
    asyncio.run(progress_middleware.on_todos_updated(todos))
    time.sleep(1)
    
    # 작업 2: 시장 데이터 분석
    print("\n" + "="*70)
    print("📊 작업 2/4: 시장 데이터 분석")
    print("="*70)
    
    asyncio.run(progress_middleware.on_task_start("시장 데이터 분석", "task_2"))
    result = analyze_data.invoke({"data_source": "AI 시장 통계 2024"})
    print(f"\n{result}")
    
    todos[1]['status'] = 'completed'
    asyncio.run(progress_middleware.on_task_complete("task_2"))
    asyncio.run(progress_middleware.on_todos_updated(todos))
    time.sleep(1)
    
    # 작업 3: 리포트 작성
    print("\n" + "="*70)
    print("📝 작업 3/4: 리포트 작성")
    print("="*70)
    
    asyncio.run(progress_middleware.on_task_start("리포트 작성", "task_3"))
    result = create_report.invoke({"title": "2024 AI 기술 트렌드 분석", "sections": 5})
    print(f"\n{result}")
    
    todos[2]['status'] = 'completed'
    asyncio.run(progress_middleware.on_task_complete("task_3"))
    asyncio.run(progress_middleware.on_todos_updated(todos))
    time.sleep(1)
    
    # 작업 4: 최종 검토
    print("\n" + "="*70)
    print("🔍 작업 4/4: 최종 검토")
    print("="*70)
    
    asyncio.run(progress_middleware.on_task_start("최종 검토", "task_4"))
    result = review_content.invoke({"content_type": "AI 트렌드 리포트"})
    print(f"\n{result}")
    
    todos[3]['status'] = 'completed'
    asyncio.run(progress_middleware.on_task_complete("task_4"))
    asyncio.run(progress_middleware.on_todos_updated(todos))
    
    print("\n" + "="*70)
    print("✅ 모든 작업 완료!")
    print("="*70)
    
    # 최종 요약
    summary = progress_middleware.get_progress_summary()
    print(f"\n📊 최종 통계:")
    print(f"  총 소요 시간: {summary['elapsed_time']:.2f}초")
    print(f"  완료된 작업: {summary['completed']}/{summary['total']}")
    print(f"  진행률: {summary['percent']:.0f}%")
    
    print(dashboard.get_summary())


# 시뮬레이션 실행
simulate_deep_agent_workflow()

## 6. 실제 Deep Agent 사용 예제

실제로 LLM과 함께 Deep Agent를 실행하는 예제입니다.
(주석 처리됨 - API 키가 있을 때 실행)

In [ ]:
# 실제 Deep Agent 실행 예제 (API 키 필요)
"""
# API 키 설정
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

# Deep Agent 실행
result = agent.invoke({
    "messages": [HumanMessage(content="""
다음 작업을 수행해주세요:

1. 2024년 생성형 AI 기술 트렌드를 조사하세요
2. 주요 기업들의 AI 투자 현황을 분석하세요
3. 조사와 분석을 바탕으로 종합 리포트를 작성하세요
4. 작성된 리포트를 검토하고 최종 버전을 완성하세요

각 단계별로 진행 상태를 업데이트해주세요.
""")]
})

print("\n최종 결과:")
print(result['messages'][-1].content)

# 진행 상태 요약
print(progress_middleware.get_progress_summary())
print(dashboard.get_summary())
"""

print("💡 실제 실행하려면 위 코드의 주석을 해제하고 API 키를 설정하세요.")

## 7. 다중 미들웨어 통합

Deep Agents의 강력함은 여러 미들웨어를 함께 사용할 수 있다는 점입니다.

In [ ]:
# 파일시스템 미들웨어와 함께 사용
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

# Store 생성 (장기 메모리용)
store = InMemoryStore()

# 여러 미들웨어를 포함한 고급 Deep Agent
advanced_agent = create_deep_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[research_topic, analyze_data, create_report, review_content],
    store=store,
    middleware=[
        # 1. Todo 관리
        TodoListMiddleware(
            system_prompt="작업 계획을 수립하고 진행 상태를 관리하세요."
        ),
        
        # 2. 파일 시스템 (단기/장기 메모리)
        FilesystemMiddleware(
            backend=lambda rt: CompositeBackend(
                default=StateBackend(rt),
                routes={"/memories/": StoreBackend(rt)}
            ),
            custom_tool_descriptions={
                "write_file": "중요한 정보를 파일로 저장하세요. /memories/ 경로는 영구 저장됩니다.",
                "read_file": "저장된 정보를 읽어오세요."
            }
        ),
        
        # 3. 진행 상태 추적 (커스텀)
        progress_middleware,
    ]
)

print("✅ 고급 Deep Agent 생성 완료")
print("\n통합된 미들웨어:")
print("  1. TodoListMiddleware - 작업 계획 및 추적")
print("  2. FilesystemMiddleware - 파일 저장 및 읽기 (단기/장기 메모리)")
print("  3. ProgressTrackingMiddleware - 실시간 진행 상태 추적")
print("\n💡 이 에이전트는 작업을 계획하고, 파일로 저장하며, 진행 상태를 실시간으로 추적합니다!")

---

## 핵심 요약

### Deep Agents의 장점

1. **모듈형 아키텍처**: 필요한 미들웨어만 선택적으로 사용
2. **기본 제공 미들웨어**: TodoList, Filesystem, SubAgent가 기본 제공
3. **확장 가능**: 커스텀 미들웨어를 쉽게 추가 가능
4. **통합성**: 여러 미들웨어가 서로 협력하여 작동

### 커스텀 미들웨어 작성 팁

1. **Middleware 베이스 클래스 상속**: Deep Agents의 표준을 따름
2. **이벤트 기반 설계**: 느슨한 결합으로 확장성 확보
3. **핸들러 패턴 활용**: 미들웨어와 UI 로직 분리
4. **비동기 지원**: async/await로 효율적인 처리

### 실무 적용

- **프로젝트 관리**: TodoList + Progress 조합
- **지식 관리**: Filesystem + 커스텀 검색 미들웨어
- **복잡한 워크플로우**: SubAgent + Progress + 커스텀 미들웨어

### 참고 자료

- [Deep Agents 공식 문서](https://docs.langchain.com/oss/python/deepagents/middleware)
- [LangGraph 문서](https://langchain-ai.github.io/langgraph/)